In [1]:
import sqlalchemy as sa
from sqlalchemy import orm



engine = sa.create_engine('sqlite:///../heroes_sql/heroes.db')
connection = engine.connect()

In [2]:
class Base(orm.DeclarativeBase):
    pass

In [3]:
class BattleParticipant(Base):
    __tablename__ = 'battle_participants'
    __table_args__ = {'extend_existing': True}

    battle_participant_id: orm.Mapped[int] = orm.mapped_column(sa.Integer, primary_key=True)
    user_id: orm.Mapped[int] = orm.mapped_column(sa.Integer, nullable=False)
    battle_id: orm.Mapped[int] = orm.mapped_column(sa.Integer, nullable=False)
    hero_id: orm.Mapped[int] = orm.mapped_column(sa.ForeignKey('heroes.hero_id'), nullable=False)


    def __repr__(self):
        return f'{type(self).__name__}({self.battle_participant_id}, {self.user_id}, {self.battle_id})'

class BattleEvents(Base):
    __tablename__ = 'battle_events'
    __table_args__ = {'extend_existing': True}

    battle_event_id: orm.Mapped[int] = orm.mapped_column(sa.Integer, primary_key=True)
    battle_event_type_id: orm.Mapped[int] = orm.mapped_column(sa.Integer, nullable=False)
    rubies_gained: orm.Mapped[int] = orm.mapped_column(sa.Integer)
    timestamp: orm.Mapped[int] = orm.mapped_column(sa.Integer, nullable=False)
    battle_participant_id: orm.Mapped[int] = orm.mapped_column(sa.ForeignKey('battle_participants.battle_participant_id'))


    def __repr__(self):
        return f'{type(self).__name__}({self.battle_participant_id}, {self.rubies_gained}, {self.timestamp})'


class Heroes(Base):
    __tablename__ = 'heroes'
    __table_args__ = {'extend_existing': True}

    name: orm.Mapped[str] = orm.mapped_column(sa.String, nullable=False, unique=True)
    hero_id: orm.Mapped[int] = orm.mapped_column(sa.Integer, primary_key=True)

    def __repr__(self):
        return f'{type(self).__name__}({self.name})'


In [4]:
# Display hero's name and amount of battle he took apart

q = (sa.select(
    Heroes.name,
    sa.func.count(BattleEvents.battle_event_id).label('TotalEvents')

).join(BattleParticipant, BattleParticipant.hero_id == Heroes.hero_id)
.join(BattleEvents, BattleEvents.battle_participant_id == BattleParticipant.battle_participant_id)
.group_by(Heroes.name)
.order_by(sa.desc('TotalEvents'))
)

dc = connection.execute(q)
print(dc.fetchall())



[('Crareek', 3687), ('Pamble', 2054), ('Huzzt', 2036), ('Tan', 1946), ('Grimm', 1820), ('Jade', 1780), ('Hillstomp', 1736), ('Mako', 1719), ('Grrdy', 1528)]


In [5]:
# Give a heroe's name who killed the highest amount of other heroes

In [6]:
from sqlalchemy.ext.automap import automap_base

In [7]:
Base = automap_base()
Base.prepare(autoload_with=engine, classname_for_table=lambda cls, table_name, table_obj: table_name.title().replace('_', ''))

In [9]:
Battles = Base.classes.Battles
BattleEvents = Base.classes.BattleEvents
Heroes = Base.classes.Heroes
Users = Base.classes.Users
BattleEventTypes = Base.classes.BattleEvents
BattleParticipants = Base.classes.BattleParticipants

In [12]:
globals()['STH'] = 1974

In [13]:
print(globals())

{'__name__': '__main__', '__doc__': 'Automatically created module for IPython interactive environment', '__package__': None, '__loader__': None, '__spec__': None, '__builtin__': <module 'builtins' (built-in)>, '__builtins__': <module 'builtins' (built-in)>, '_ih': ['', "import sqlalchemy as sa\nfrom sqlalchemy import orm\nengine = sa.create_engine('sqlite:///../heroes_sql/heroes.db')\nconnection = engine.connect()", 'class Base(orm.DeclarativeBase):\n    pass', "class BattleParticipant(Base):\n    __tablename__ = 'battle_participants'\n    __table_args__ = {'extend_existing': True}\n\n    battle_participant_id: orm.Mapped[int] = orm.mapped_column(sa.Integer, primary_key=True)\n    user_id: orm.Mapped[int] = orm.mapped_column(sa.Integer, nullable=False)\n    battle_id: orm.Mapped[int] = orm.mapped_column(sa.Integer, nullable=False)\n    hero_id: orm.Mapped[int] = orm.mapped_column(sa.ForeignKey('heroes.hero_id'), nullable=False)\n\n\n    def __repr__(self):\n        return f'{type(self)

In [ ]:
q = (sa.select(
    Heroes.name,
    sa.func.count(BattleEvents.battle_event_id).label('TotalKills')

).join(BattleParticipant, BattleParticipant.hero_id == Heroes.hero_id)
.join(BattleEvents, BattleEvents.battle_participant_id == BattleParticipant.battle_participant_id)
.join(BattleEventTypes, BattleEventTypes.battle_event_type_id == BattleEvents.battle_event_type_id)
.where(BattleEventTypes.name == 'HERO_KILL')
.group_by(Heroes.name)
.order_by(sa.desc('TotalKills'))
)

dc = connection.execute(q)
print(dc.fetchall())